# Gold Layer — MovieLens Aggregations

**Purpose:** Read clean Silver tables and produce business-level aggregation tables for dashboards and analysis.

Gold tables are pre-aggregated and optimised for fast reads — analysts and BI tools query Gold, never Bronze or Silver directly.

| Table | Description |
|-------|-------------|
| `gold_top_movies` | Best-rated movies with > 100 ratings, sorted by avg rating |
| `gold_ratings_by_year` | Total movies and avg rating grouped by release year |
| `gold_genre_popularity` | Avg rating and total ratings per genre, sorted by volume |
| `gold_user_activity` | Rating count and avg rating per active user (> 10 ratings) |

## Config & Imports

In [ ]:
from pyspark.sql.functions import (
    avg,
    col,
    count,
    countDistinct,
    explode,
    round,
)

# Source: clean Silver Delta tables
SOURCE_SCHEMA = "workspace.silver"

# Target: Gold aggregation tables consumed by dashboards / ML
TARGET_SCHEMA = "workspace.gold"

## Step 1 — Create Gold Schema

`IF NOT EXISTS` makes this safe to re-run at any time.

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

print(f"Schema ready: {TARGET_SCHEMA}")

## Step 2 — Read Silver Tables

All four Gold aggregations are built from these two Silver tables.

In [ ]:
silver_ratings_df = spark.table(f"{SOURCE_SCHEMA}.silver_ratings")
silver_movies_df  = spark.table(f"{SOURCE_SCHEMA}.silver_movies")

print(f"silver_ratings : {silver_ratings_df.count():>12,} rows")
print(f"silver_movies  : {silver_movies_df.count():>12,} rows")

## Table 1 — gold_top_movies

Join ratings with movie titles, then aggregate per movie.
The 100-rating threshold removes niche titles that would skew the top of the list
(a movie with one 5-star rating would otherwise rank #1).

In [ ]:
top_movies_df = (
    silver_ratings_df
    # Bring in the movie title — only need movieId + title from movies
    .join(silver_movies_df.select("movieId", "title"), on="movieId", how="inner")
    .groupBy("movieId", "title")
    .agg(
        round(avg("rating"), 2).alias("avg_rating"),    # rounded to 2 decimal places
        count("rating").alias("total_ratings"),
    )
    .filter(col("total_ratings") > 100)    # exclude movies with too few ratings
    .orderBy(col("avg_rating").desc())
)

print("Writing gold_top_movies ...")
(
    top_movies_df
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TARGET_SCHEMA}.gold_top_movies")
)
print(f"  Done — {top_movies_df.count():,} movies with > 100 ratings")
top_movies_df.show(5, truncate=False)

## Table 2 — gold_ratings_by_year

Group movies by release year to show how the catalogue and rating patterns evolved over time.
`countDistinct` on movieId gives unique movies per year (a movie appears once per rating row after the join).

In [ ]:
ratings_by_year_df = (
    silver_movies_df
    .filter(col("release_year").isNotNull())    # skip movies with no year in their title
    # Left join: keep all movies even if they have no ratings yet
    .join(silver_ratings_df.select("movieId", "rating"), on="movieId", how="left")
    .groupBy("release_year")
    .agg(
        countDistinct("movieId").alias("total_movies"),  # unique movies that year
        round(avg("rating"), 2).alias("avg_rating"),     # avg of all ratings for that year
    )
    .orderBy("release_year")
)

print("Writing gold_ratings_by_year ...")
(
    ratings_by_year_df
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TARGET_SCHEMA}.gold_ratings_by_year")
)
print(f"  Done — {ratings_by_year_df.count():,} years")
ratings_by_year_df.show(5, truncate=False)

## Table 3 — gold_genre_popularity

`explode()` turns each movie's `["Action", "Adventure"]` array into one row per genre,
so a movie tagged with 3 genres contributes to all 3 genre aggregations.
Sorted by `total_ratings` descending to show the most-engaged genres first.

In [ ]:
genre_popularity_df = (
    silver_movies_df
    # explode turns ["Action", "Adventure"] → two rows, one per genre
    .select("movieId", explode("genres").alias("genre"))
    # Join with ratings so each genre row gets a rating value
    .join(silver_ratings_df.select("movieId", "rating"), on="movieId", how="inner")
    .groupBy("genre")
    .agg(
        round(avg("rating"), 2).alias("avg_rating"),
        count("rating").alias("total_ratings"),
    )
    .orderBy(col("total_ratings").desc())
)

print("Writing gold_genre_popularity ...")
(
    genre_popularity_df
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TARGET_SCHEMA}.gold_genre_popularity")
)
print(f"  Done — {genre_popularity_df.count():,} genres")
genre_popularity_df.show(truncate=False)

## Table 4 — gold_user_activity

Profile each user by how much they rate and how generous their scores are.
The 10-rating threshold removes one-time users whose data is statistically unreliable.

In [ ]:
user_activity_df = (
    silver_ratings_df
    .groupBy("userId")
    .agg(
        count("rating").alias("total_ratings"),      # how many movies this user rated
        round(avg("rating"), 2).alias("avg_rating"), # this user's average score
    )
    .filter(col("total_ratings") > 10)    # only active users
    .orderBy(col("total_ratings").desc())
)

print("Writing gold_user_activity ...")
(
    user_activity_df
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{TARGET_SCHEMA}.gold_user_activity")
)
print(f"  Done — {user_activity_df.count():,} users with > 10 ratings")
user_activity_df.show(5, truncate=False)

## Row Count Summary

Read back from all four Gold Delta tables on disk to confirm everything landed correctly.

In [ ]:
gold_tables = [
    "gold_top_movies",
    "gold_ratings_by_year",
    "gold_genre_popularity",
    "gold_user_activity",
]

print("=" * 50)
print("  Gold aggregations complete")
print("=" * 50)

for table in gold_tables:
    n = spark.table(f"{TARGET_SCHEMA}.{table}").count()
    print(f"  {table:<25} : {n:>8,} rows")

print("=" * 50)